In [1]:
import re
import gc
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm import tqdm

import spacy
import scispacy
import medspacy

notes = pd.read_csv(
    "../outputs/eda_base_dataset.csv"
)

print("Dataset shape:", notes.shape)


Dataset shape: (11061, 9)


In [13]:
print("Loading NLP model...")

nlp = spacy.load(
    "en_core_sci_md",
    disable=[
        "tagger",
        "parser",
        "attribute_ruler",
        "lemmatizer"
    ]
)

nlp.add_pipe("sentencizer")
nlp.add_pipe("medspacy_context")

print("\nPipeline loaded:")
print(nlp.pipe_names)

Loading NLP model...

Pipeline loaded:
['tok2vec', 'ner', 'sentencizer', 'medspacy_context']


In [14]:
def clean_note(text):

    if pd.isnull(text):
        return ""

    text = str(text)

    text = re.sub(
        r"\[\*\*.*?\*\*\]",
        " ",
        text
    )

    text = re.sub(r"[_\-]{2,}", " ", text)

    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\r+", " ", text)
    text = re.sub(r"\t+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.lower().strip()

print("\nCleaning notes...")

tqdm.pandas()

notes["clean_text"] = notes["TEXT"].progress_apply(clean_note)

print("Cleaning complete.")



Cleaning notes...


100%|██████████| 11061/11061 [00:05<00:00, 1940.86it/s]

Cleaning complete.


In [15]:
before = len(notes)

notes = notes[
    notes["clean_text"].str.len() > 50
].copy()

after = len(notes)

print(f"\nRemoved {before - after:,} short notes.")


Removed 0 short notes.


In [16]:
print("\n=== ENTITY LABEL DEBUG ===\n")

sample_text = notes.iloc[0]["clean_text"][:5000]
print(sample_text)

doc = nlp(sample_text)

print("Detected entities:\n")

for ent in doc.ents[:30]:
    print(ent.text, " --> ", ent.label_)

print("\nTotal entities:", len(doc.ents))



=== ENTITY LABEL DEBUG ===

name: , a unit no: admission date: discharge date: date of birth: sex: f service: addendum: the patient had to spend last night ( ) because of unable to transport in the evening and required dialysis during the day. this morning, she is doing well. she was found asleep. on examination, her abdomen was soft, nontender, and nondistended. she was afebrile. she was tolerating oral intake. she was complaining of nausea, but no vomiting. she was able to eat regular meals (a full meal). condition on discharge: same. discharge disposition: the patient to be discharged back to nursing facility today. , dictated by: medquist36 d: 07:46:00 t: 11:48:41 job#:
Detected entities:

admission  -->  ENTITY
discharge date  -->  ENTITY
birth  -->  ENTITY
sex  -->  ENTITY
service  -->  ENTITY
addendum  -->  ENTITY
patient  -->  ENTITY
night  -->  ENTITY
transport  -->  ENTITY
evening  -->  ENTITY
dialysis  -->  ENTITY
day  -->  ENTITY
morning  -->  ENTITY
asleep  -->  ENTITY
ex

In [ ]:
def extract_symptoms(doc):
    symptoms = []
    for ent in doc.ents:
        try:
            context = ent._.context_attributes
            is_negated = context.get(
                "is_negated",
                False
            )
            is_uncertain = context.get(
                "is_uncertain",
                False
            )
        except:
            is_negated = False
            is_uncertain = False

        if is_negated:
            continue

        if is_uncertain:
            continue

        text = ent.text.strip().lower()

        if len(text) < 3:
            continue

        if text.isnumeric():
            continue

        symptoms.append(text)

    return symptoms

def process_chunk(text_chunk):
    results = []

    pipe = nlp.pipe(
        text_chunk,
        batch_size=16,
        n_process=1
    )

    for doc in pipe:

        symptoms = extract_symptoms(doc)

        results.append(symptoms)

    return results

In [18]:
CHUNK_SIZE = 1000

all_symptoms = []

texts = notes["clean_text"]

print("\nRunning clinical NER...\n")

for start in tqdm(
    range(0, len(texts), CHUNK_SIZE)
):

    end = start + CHUNK_SIZE

    chunk = texts.iloc[start:end]

    chunk_results = process_chunk(chunk)

    all_symptoms.extend(chunk_results)

    del chunk_results
    gc.collect()

notes["symptoms"] = all_symptoms

print("\nNER extraction complete.")


Running clinical NER...



100%|██████████| 12/12 [17:43<00:00, 88.64s/it]


NER extraction complete.
